# DynamoDB

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-interview-questions/blob/master/notebooks/topics/dynamodb.ipynb)

*How to use this notebook.* Read the explanation cells and run the code cells below them. Python examples list their imports, so install what a section needs before running it. SQL and shell snippets are illustrative: they show the syntax for the platform named in that section and need that platform (or a compatible engine) to run.

## How do you model data in DynamoDB?

DynamoDB modeling starts with access patterns, not with entities. You write down every query the application must run, then choose a primary key, secondary indexes, and item collections that satisfy those queries. Unlike relational modeling, you denormalize aggressively and design multiple item types in one table. There are no joins and no server-side aggregations beyond `Query` on a key and `Scan` on everything.

A practical process:

1. List access patterns with their frequency and latency target, for example "get order by id (1000/s)" or "list a customer's orders by recency (50/s)".
2. Choose the partition key for even write distribution and the sort key for range queries and item collections.
3. Add a GSI only for access patterns the base key cannot serve.
4. Decide the item types that share the table and give them a key prefix convention.

Item collection (single-table) key pattern:

```text
PK = CUSTOMER#123
SK = PROFILE
SK = ORDER#2026-10-04#999
```


```json
{
  "PK": {"S": "CUSTOMER#123"},
  "SK": {"S": "ORDER#2026-10-04#999"},
  "entity_type": {"S": "order"},
  "order_id": {"S": "999"},
  "status": {"S": "shipped"},
  "amount": {"N": "149.90"},
  "order_ts": {"S": "2026-10-04T12:31:00Z"},
  "GSI1PK": {"S": "CUSTOMER#123#STATUS#shipped"},
  "GSI1SK": {"S": "2026-10-04T12:31:00Z"}
}
```

`PK`/`SK` are generic key attributes that can hold any entity type, and the `CUSTOMER#` prefix makes the value sortable and greppable. Querying `PK = CUSTOMER#123` returns the profile and every order in one request. Adding a GSI on `GSI1PK`/`GSI1SK` answers "all shipped orders for this customer, newest first" without a scan.

Interview guidance:

+ Model for the query you will run thousands of times per second, not for the shape of the CSV you imported once.
+ Prefer a small number of tables in a service. True single-table design is powerful but harder to operate; multi-table is acceptable when access patterns are cleanly separated.
+ If you cannot name the access pattern that a new index serves, do not create it. Every index increases write cost and failure surface.
+ DynamoDB is eventually consistent by default on reads and GSIs; request `ConsistentRead=True` only on the base table when the pattern truly needs it.

Mental model: DynamoDB is a hash table of items. DynamoDB hashes the partition key to choose a physical partition, then stores every item that shares that partition key together, ordered by the sort key. That group of items is one item collection, and the whole collection can be read with a single `Query`. The sort key gives the collection an order, which is what makes `begins_with`, `between`, and `ScanIndexForward=False` cheap.

How it works: as a table grows, DynamoDB splits partitions and moves key ranges so that storage and traffic stay spread. You never choose a partition directly, and the partition key value never changes where an item logically lives. A GSI copies the attributes you project into a second keyspace with its own partitions, so an index has its own heat and its own throughput even when the base table is evenly loaded. A sparse GSI indexes only items that carry the index key, so leaving that key off most items keeps the index small and cheap.

Worked example: `PK = CUSTOMER#123` and `SK = ORDER#2026-10-04#999`. The customer is the partition, so the profile and every order for that customer are one collection and one query; the date-prefixed sort key gives recency order for free. Common mistake: giving each entity its own table and joining in the application, which turns one `Query` into N round trips and loses the atomicity you get from co-locating items.

## What are partition keys, sort keys, GSIs, and LSIs?

The partition key (hash key) is hashed to a physical partition and determines data distribution. The sort key (range key) orders items inside a partition and enables range queries and `begins_with`/`between` conditions. Partition key plus sort key must be unique. A global secondary index (GSI) defines a completely different partition/sort key pair and is maintained asynchronously. A local secondary index (LSI) keeps the table's partition key but uses a different sort key, and must be created with the table.

Example schema for an orders workload:

```text
table key:   customer_id (PK), order_ts (SK)
GSI "by_status":  order_status (PK), order_ts (SK)
LSI "by_amount":  customer_id (PK), amount (SK)
```

```json
{
  "TableName": "orders",
  "KeySchema": [
    {"AttributeName": "customer_id", "KeyType": "HASH"},
    {"AttributeName": "order_ts", "KeyType": "RANGE"}
  ],
  "AttributeDefinitions": [
    {"AttributeName": "customer_id", "AttributeType": "S"},
    {"AttributeName": "order_ts", "AttributeType": "S"},
    {"AttributeName": "order_status", "AttributeType": "S"}
  ],
  "GlobalSecondaryIndexes": [
    {
      "IndexName": "by_status",
      "KeySchema": [
        {"AttributeName": "order_status", "KeyType": "HASH"},
        {"AttributeName": "order_ts", "KeyType": "RANGE"}
      ],
      "Projection": {"ProjectionType": "ALL"}
    }
  ]
}
```

Key differences that interviewers probe:

| Aspect | GSI | LSI |
|---|---|---|
| Partition key | Any attribute, independent of table | Must equal the table partition key |
| Creation | Any time, online | Only at table creation |
| Consistency | Eventually consistent only | Supports strongly consistent reads |
| Capacity | Own RCU/WCU; writes fan out to the index | Shares table throughput |
| Partition limit | No fixed item-collection size limit | 10 GB per partition-key item collection |
| Limit | 20 GSIs per table (default quota) | 5 LSIs per table |

Design notes:

+ Indexes are updated asynchronously. A read immediately after a base-table write can be eventually consistent and, in the case of a GSI, can miss the just-written item for a short window.
+ Use sparse indexes deliberately: if items that should be indexed carry the index key and others omit it, the others never appear in the index, keeping it small and cheap.
+ The 10 GB LSI ceiling is per partition key, not per table. A hot `customer_id` with millions of orders can hit it; a GSI has no equivalent per-key ceiling but can still throttle on its own partition.

How an index is maintained: a GSI is a second table that DynamoDB updates for you. A base write that adds, changes, or removes a projected attribute is replicated to the index asynchronously, so a GSI read is eventually consistent and can briefly miss a just-written item. An LSI shares the base table's partition and data, uses a different sort key over the same items, and supports strongly consistent reads, which is why it is the only index that can back a read-your-writes pattern on a non-key attribute.

Capacity and limits: each GSI has its own RCU and WCU, or its own billed requests in on-demand mode, and its own hot-partition behaviour; provisioned capacity for an index is set separately from the table. By default a table may have up to 20 GSIs and 5 LSIs. An LSI adds a 10 GB ceiling per partition-key item collection that a table without an LSI does not have. Common mistake: treating an LSI as a free index and then hitting `ItemCollectionSizeLimitExceededException` on a hot partition key, or assuming a GSI query can be strongly consistent. Neither can be fixed later without a new index or a new table.

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
# Requires boto3 and AWS credentials; shown for review.
import boto3

client = boto3.client("dynamodb", region_name="us-east-1")

client.create_table(
    TableName="orders",
    KeySchema=[
        {"AttributeName": "customer_id", "KeyType": "HASH"},
        {"AttributeName": "order_ts", "KeyType": "RANGE"},
    ],
    AttributeDefinitions=[
        {"AttributeName": "customer_id", "AttributeType": "S"},
        {"AttributeName": "order_ts", "AttributeType": "S"},
        {"AttributeName": "order_status", "AttributeType": "S"},
        {"AttributeName": "amount", "AttributeType": "N"},
    ],
    LocalSecondaryIndexes=[
        {
            "IndexName": "by_amount",
            "KeySchema": [
                {"AttributeName": "customer_id", "KeyType": "HASH"},
                {"AttributeName": "amount", "KeyType": "RANGE"},
            ],
            "Projection": {"ProjectionType": "KEYS_ONLY"},
        }
    ],
    GlobalSecondaryIndexes=[
        {
            "IndexName": "by_status",
            "KeySchema": [
                {"AttributeName": "order_status", "KeyType": "HASH"},
                {"AttributeName": "order_ts", "KeyType": "RANGE"},
            ],
            "Projection": {"ProjectionType": "INCLUDE", "NonKeyAttributes": ["amount"]},
        }
    ],
    BillingMode="PAY_PER_REQUEST",
)


## How do you avoid hot partitions in DynamoDB?

A hot partition happens when one partition key value receives a disproportionate share of traffic. DynamoDB partitions have throughput limits, and even on-demand tables throttle a single very hot key because the limit is per physical partition, not per table. Common causes are a low-cardinality partition key (a constant tenant, today's date, a status value) and monotonically increasing keys.

Bad versus sharded keys:

```text
bad:    PK = EVENT_DATE#2026-10-04          # every event lands in one partition
better: PK = EVENT_DATE#2026-10-04#BUCKET#07  # 16-64 random buckets spread writes
```

In [ ]:
import random

def shard_key(event_date: str, buckets: int = 16) -> str:
    bucket = random.randint(0, buckets - 1)
    return f"EVENT_DATE#{event_date}#BUCKET#{bucket:02d}"

The trade-off is that reads must query every bucket and merge results, which multiplies RCU and latency.

Other mitigations:

+ Use write sharding with a random or hash suffix on the hot key and accept fan-out reads.
+ Use a high-cardinality key such as `order_id`, `device_id`, or `customer_id` instead of a status or a date.
+ Cache the hot item (DAX or an application cache) so read traffic does not reach the partition.
+ Move counters to a distributed pattern: write one update per shard, then sum on read, or use DynamoDB Streams plus aggregations.
+ Watch `ThrottledRequests`, `WriteThrottleEvents`, and `ReadThrottleEvents` in CloudWatch, and per-key Contributor Insights to find the offending key.
+ On provisioned tables, `AdaptiveCapacity` automatically shifts capacity toward hot partitions, but it cannot fix a key so hot that it exceeds a single partition's ceiling.

Why throttling is per partition: each partition serves up to 3,000 RCU and 1,000 WCU. A table's provisioned capacity is a budget spread across its partitions, not a per-key guarantee, so if one key receives more than its share that partition throttles even when the table has spare capacity overall. Adaptive capacity moves capacity toward busy partitions and can isolate a frequently accessed item so it gets up to the full partition ceiling, but it cannot exceed that ceiling, and it will not split an item collection that has an LSI.

Worked example: 10,000 writes/s of 1 KB to a table keyed by calendar date. One date value is one partition and absorbs about 1,000 writes/s, so it throttles at a tenth of the offered load. Sharding the key into 16 buckets spreads roughly 625 writes/s per bucket and stays under the ceiling; reads then have to query all 16 buckets in parallel and merge the results.

Common mistake: salting the partition key but leaving the read path as a `Scan`, which recovers write throughput and gives away read cost. Derive the bucket deterministically from an attribute the query already knows, for example `crc32(user_id) % 16`, so an identity read targets one bucket and only a true aggregate fans out. Burst capacity banks unused capacity for up to about five minutes to smooth short spikes, but it is not a fix for sustained skew.

In [ ]:
PARTITION_WCU = 1_000


def shard_load(total_writes_per_second, buckets):
    return total_writes_per_second / buckets


def under_limit(total_writes_per_second, buckets):
    return shard_load(total_writes_per_second, buckets) <= PARTITION_WCU


for buckets in (1, 8, 16, 32):
    print(f"{buckets:>2} buckets -> {shard_load(25_000, buckets):7.1f} WCU per bucket, "
          f"ok={under_limit(25_000, buckets)}")

assert not under_limit(25_000, 16)
assert under_limit(25_000, 32)


## How do you read and write items with boto3?

boto3 offers two styles. The low-level `client` maps directly to the DynamoDB API and uses typed `AttributeValue` JSON. The higher-level `resource` (or `Table`) accepts and returns plain Python values and is easier to write but hides wire details you still need to recognize in interviews.

Low-level client with DynamoDB JSON:

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import boto3

client = boto3.client("dynamodb", region_name="us-east-1")

client.put_item(
    TableName="orders",
    Item={
        "PK": {"S": "CUSTOMER#123"},
        "SK": {"S": "ORDER#2026-10-04#999"},
        "order_id": {"S": "999"},
        "status": {"S": "shipped"},
        "amount": {"N": "149.90"},
        "tags": {"SS": ["priority", "gift"]},
        "meta": {"M": {"channel": {"S": "mobile"}}},
    },
    ConditionExpression="attribute_not_exists(PK)",
)

response = client.get_item(
    TableName="orders",
    Key={"PK": {"S": "CUSTOMER#123"}, "SK": {"S": "ORDER#2026-10-04#999"}},
    ConsistentRead=True,
)
item = response.get("Item")

Higher-level resource with plain values, plus a typed deserializer if you mix styles:

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import boto3
from boto3.dynamodb.conditions import Key

table = boto3.resource("dynamodb", region_name="us-east-1").Table("orders")

table.put_item(Item={
    "PK": "CUSTOMER#123",
    "SK": "ORDER#2026-10-04#999",
    "status": "shipped",
    "amount": 149.90,
})

page = table.query(
    KeyConditionExpression=Key("PK").eq("CUSTOMER#123") & Key("SK").begins_with("ORDER#"),
    ProjectionExpression="order_id, #s, amount",
    ExpressionAttributeNames={"#s": "status"},
    Limit=25,
)

AWS CLI for a quick smoke test:

```bash
aws dynamodb put-item \
  --table-name orders \
  --item '{"PK":{"S":"CUSTOMER#123"},"SK":{"S":"ORDER#2026-10-04#999"},"status":{"S":"shipped"}}'

aws dynamodb get-item \
  --table-name orders \
  --key '{"PK":{"S":"CUSTOMER#123"},"SK":{"S":"ORDER#2026-10-04#999"}}'
```

PartiQL is useful for ad-hoc reads in the console and for teams that want SQL-like syntax:

```sql
SELECT order_id, status, amount
FROM orders
WHERE PK = 'CUSTOMER#123' AND begins_with(SK, 'ORDER#')
```

Notes:

+ `PutItem` replaces the whole item. Use `UpdateItem` with `SET`/`ADD`/`REMOVE` to change individual attributes safely under concurrency.
+ Attribute names that collide with reserved words (`status`, `size`, `timestamp`) must be aliased with `ExpressionAttributeNames`, which is why `#s` appears above.
+ `ProjectionExpression` reduces network and RCU cost by returning only the attributes you need.
+ `boto3` retries transient errors automatically; do not disable that except in tests that need deterministic failure behavior.

Client versus resource: the client is a thin wrapper over the wire API and speaks DynamoDB JSON such as `{"S": "..."}` and `{"N": "..."}`, which is exactly what appears in CloudTrail, CloudWatch Logs, and the raw API. The resource maps that JSON to Python values, using `Decimal` for numbers and `set` for `SS` and `NS`, and exposes condition objects from `boto3.dynamodb.conditions`. Both retry transient failures, and the resource does not change consistency semantics.

Batch operations cut round trips, not capacity: `BatchWriteItem` takes up to 25 put or delete requests and `BatchGetItem` up to 100 keys and 16 MB total, and every item still costs its normal WCU or RCU. Both calls can partially fail and return `UnprocessedItems`, so production code drains that map with backoff. The resource's `batch_writer` handles batching and unprocessed retries for you, subject to the same 25-item limit per underlying call.

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
# Requires boto3 and AWS credentials; shown for review.
import boto3

table = boto3.resource("dynamodb", region_name="us-east-1").Table("orders")

with table.batch_writer(overwrite_by_pkeys=["PK", "SK"]) as batch:
    for n in range(100):
        batch.put_item(Item={
            "PK": "CUSTOMER#123",
            "SK": f"ORDER#2026-10-04#{n:04d}",
            "status": "created",
            "amount": 10.0 + n,
        })


The same operations from the command line, including PartiQL for ad-hoc reads:

```bash
# Batch write from a JSON request file (25 items per call).
aws dynamodb batch-write-item --request-items file://orders-batch.json

# PartiQL with a bound parameter.
aws dynamodb execute-statement \
  --statement "SELECT order_id, status, amount FROM orders WHERE PK = ?" \
  --parameters '[{"S":"CUSTOMER#123"}]'

# PartiQL write.
aws dynamodb execute-statement \
  --statement "UPDATE orders SET status = 'shipped' WHERE PK = ? AND SK = ?" \
  --parameters '[{"S":"CUSTOMER#123"},{"S":"ORDER#2026-10-04#999"}]'
```

PartiQL is convenient for ad-hoc work and small scripts, but the cost model is unchanged: a `SELECT` without a key condition is a `Scan`, and each statement round trip is still a request.

## What is the difference between Query and Scan?

`Query` seeks directly to one partition (and optionally a sort-key range) using the index, so cost and latency scale with the number of matching items. `Scan` reads the entire table or index and then filters, so cost scales with table size regardless of how many rows you keep. Scan is the classic interview trap: it looks convenient and silently becomes the most expensive operation in the account.

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import boto3
from boto3.dynamodb.conditions import Attr, Key

table = boto3.resource("dynamodb", region_name="us-east-1").Table("orders")

# Good: one partition, one sort-key range
resp = table.query(
    KeyConditionExpression=Key("PK").eq("CUSTOMER#123") & Key("SK").between(
        "ORDER#2026-01-01", "ORDER#2026-10-04"
    ),
    FilterExpression=Attr("amount").gt(100),
    ScanIndexForward=False,
)

# Bad by default: full scan with a filter
resp = table.scan(
    FilterExpression=Attr("status").eq("cancelled"),
)

Pagination is identical for both and is driven by `LastEvaluatedKey`; a single call returns at most 1 MB:

In [ ]:
def query_all(table, **kwargs):
    items = []
    while True:
        resp = table.query(**kwargs)
        items.extend(resp["Items"])
        last = resp.get("LastEvaluatedKey")
        if not last:
            return items
        kwargs["ExclusiveStartKey"] = last

Interview notes:

+ `FilterExpression` is applied after the read. You still pay for the items that were read and discarded, so a filter that removes 99% of rows still costs 100%.
+ Use `Limit` for page size, not as a cost control; it caps items scanned per call, not total work.
+ If a scan is genuinely needed (one-off backfill, small config table, analytics export), isolate it, run it off-peak, and consider `ParallelScan` with `TotalSegments`.
+ If you find yourself scanning to answer an access pattern, the model is wrong: add a GSI or restructure keys.

How the read path works: `Query` hashes the partition key to one partition, walks the sort-key index inside it, and stops as soon as the key condition no longer matches. Cost scales with the items it visits in that collection, not with table size. `Scan` reads every item in every partition and applies the filter afterwards, so its cost scales with the table. A single call returns at most 1 MB, and both operations bill for items read before the filter runs.

Worked example: a 1 KB strongly consistent read costs 1 RCU whether it comes from a `Query` or a `Scan`; the difference is how many items are touched. A `Query` that visits 25 items costs roughly 25 RCU units, while a `Scan` over 5,000,000 items visits all of them. Pagination uses `LastEvaluatedKey`, and `ParallelScan` splits the table into segments for speed but does not reduce total RCU.

In [ ]:
def ceil_div(numerator, denominator):
    return -(-numerator // denominator)


def rcu(item_kb, strongly_consistent=True):
    units = ceil_div(item_kb, 4)
    return units if strongly_consistent else units / 2


query_items, scan_items = 25, 5_000_000
print("query, eventually consistent RCU:", query_items * rcu(1, False))
print("scan,  eventually consistent RCU:", scan_items * rcu(1, False))
assert query_items * rcu(1, False) < scan_items * rcu(1, False)


## What are DynamoDB capacity modes and how do you control cost?

DynamoDB offers on-demand and provisioned capacity. Cost comes from writes, reads, storage, indexes, backups, and streams, and the read/write unit accounting is the part candidates usually miss.

Unit math to memorize:

+ 1 WCU = 1 write per second for an item up to 1 KB; larger items consume `ceil(size_kb)` WCUs.
+ 1 RCU = 1 strongly consistent read per second for an item up to 4 KB, or 2 eventually consistent reads; larger items consume `ceil(size_kb / 4)`.
+ A GSI multiplies write cost: every base write that changes an indexed attribute writes to the index too.
+ Transactions cost roughly double (prepare plus commit), and transactional reads cost 2 RCUs per 4 KB.

```bash
# On-demand table (no capacity planning, pay per request)
aws dynamodb create-table --table-name orders \
  --attribute-definitions AttributeName=PK,AttributeType=S AttributeName=SK,AttributeType=S \
  --key-schema AttributeName=PK,KeyType=HASH AttributeName=SK,KeyType=RANGE \
  --billing-mode PAY_PER_REQUEST

# Provisioned table with application auto scaling
aws dynamodb update-table --table-name orders \
  --billing-mode PROVISIONED \
  --provisioned-throughput ReadCapacityUnits=50,WriteCapacityUnits=50
```

```yaml
# Application Auto Scaling target for the orders table
Targets:
  - Id: orders-writes
    ServiceNamespace: dynamodb
    ScalableDimension: dynamodb:table:WriteCapacityUnits
    ResourceId: table/orders
    MinCapacity: 20
    MaxCapacity: 2000
```

Choosing a mode:

+ On-demand is right when traffic is spiky, new, or hard to forecast. It scales immediately and you pay per request; it can be several times more expensive at steady high utilization.
+ Provisioned is right for predictable, sustained traffic. You can reserve capacity for large discounts, and auto scaling reacts within minutes, not instantly.
+ There is also provisioned with reserved capacity for one- or three-year commitments when the baseline is stable.

Cost controls that matter:

+ Reduce item size; RCU/WCU are rounded up per KB, so a 1.1 KB item costs twice a 1.0 KB item's write.
+ Use eventually consistent reads where acceptable, which halves read cost.
+ Project fewer attributes into GSIs; `KEYS_ONLY` and `INCLUDE` cost less than `ALL`.
+ Enable TTL to expire old items for free instead of paying to delete and store them.
+ Avoid scans over large tables; a bad scan can dominate the monthly bill.
+ Turn on DynamoDB Streams only where you consume them, and monitor index storage growth separately.

The units: 1 WCU is one write per second for an item up to 1 KB, and item size rounds up to the next KB, so a 1.1 KB write costs 2 WCU. 1 RCU is one strongly consistent read per second for an item up to 4 KB, rounding up per 4 KB, and an eventually consistent read costs half. The rounding is why shrinking an item from 1.1 KB to 1.0 KB halves its write cost, and why a 4.1 KB item costs two read units where a 4.0 KB item costs one.

Modes and how they bill: provisioned capacity is reserved per second and billed per hour whether or not you use it, with Application Auto Scaling adjusting it on a target-utilisation policy that reacts in minutes. On-demand bills per request unit with no reservation and scales within seconds, so it wins on spiky or unpredictable traffic and loses on a steady, high-utilisation baseline. A stable floor can be put on reserved capacity for a one- or three-year commitment.

Where the capacity goes: writes fan out to every index that projects a changed attribute, a transactional write costs roughly double the WCU, and reads through a GSI consume the index's capacity. Adaptive capacity plus up to about five minutes of burst capacity absorb short imbalances. On-demand tables also have a default table-level quota of 40,000 read and 40,000 write request units that can be raised, and optional maximum-throughput settings to cap spend.

Worked example: 500 writes/s of 2 KB is `ceil(2) x 500 = 1,000` WCU; 800 reads/s of 6 KB eventually consistent is `ceil(6/4) = 2` units halved to 1, so 800 RCU. Common mistake: sizing on average item size and average traffic. Use the p99 item size and the peak second, because the rounding and the per-second limit apply to each individual request and second.

In [ ]:
def ceil_div(numerator, denominator):
    return -(-numerator // denominator)


def write_units(item_bytes, writes_per_second):
    return ceil_div(item_bytes, 1024) * writes_per_second


def read_units(item_bytes, reads_per_second, strongly_consistent=True):
    units = ceil_div(item_bytes, 4096)
    return units * reads_per_second if strongly_consistent else units * reads_per_second / 2


print("WCU for 2 KB x 500 writes/s:", write_units(2048, 500))
print("RCU for 6 KB x 800 eventual reads/s:", read_units(6144, 800, False))
assert write_units(2048, 500) == 1000
assert read_units(6144, 800, False) == 800


Guardrails for on-demand spend, and the same table switched to provisioned:

```bash
# Cap an on-demand table so a runaway job cannot bill without limit.
aws dynamodb update-table --table-name orders \
  --billing-mode PAY_PER_REQUEST \
  --on-demand-throughput MaxReadRequestUnits=20000,MaxWriteRequestUnits=8000

# Provisioned, with the capacity of each GSI set separately.
aws dynamodb update-table --table-name orders \
  --billing-mode PROVISIONED \
  --provisioned-throughput ReadCapacityUnits=50,WriteCapacityUnits=50 \
  --global-secondary-index-updates \
  '[{"Update":{"IndexName":"by_status","ProvisionedThroughput":{"ReadCapacityUnits":25,"WriteCapacityUnits":25}}}]'
```


## How do GSI and LSI projections and limits affect design?

Projection decides which attributes are copied into the index. It controls index size, index write cost, and whether a query can be answered without a follow-up read of the base table.

```json
{
  "IndexName": "by_status",
  "KeySchema": [
    {"AttributeName": "order_status", "KeyType": "HASH"},
    {"AttributeName": "order_ts", "KeyType": "RANGE"}
  ],
  "Projection": {
    "ProjectionType": "INCLUDE",
    "NonKeyAttributes": ["amount", "customer_id"]
  }
}
```

Projection types:

+ `KEYS_ONLY`: smallest and cheapest, but every query needs a second `GetItem` (or batch) to fetch other attributes.
+ `INCLUDE`: a limited list of extra attributes; a good middle ground.
+ `ALL`: no follow-up read, but the highest storage and write amplification.

Operational rules:

+ You cannot change an index's key schema later; changing projections requires a new index and a backfill. Plan key schemas carefully up front.
+ You can add a GSI online to a live table, which is the usual migration path when a new access pattern appears.
+ Queries that fetch non-projected attributes turn one request into N follow-up reads (the "N+1" DynamoDB anti-pattern). Prefer `ALL` or `INCLUDE` when the access pattern is hot.
+ Sparse indexes are a feature: only items that contain the index key are projected, so conditional keys keep an index small.
+ A GSI has its own throughput and its own hot-partition behavior; a low-cardinality GSI partition key can throttle even when the base table is healthy.

What a projection costs: the projection decides which attributes are copied into the index. Key attributes are always projected. `KEYS_ONLY` copies only keys, `INCLUDE` copies the listed non-key attributes, and `ALL` copies the whole item. Every base-table write that changes a projected attribute consumes index WCU as well, so `ALL` on a 4 KB item can double the write cost of the table, while `KEYS_ONLY` on a 200-byte index adds little.

Worked example: 1,000 writes/s of a 1.5 KB item cost 2,000 WCU on the base table. Adding an `ALL` GSI of the same size adds 2,000 more; a `KEYS_ONLY` GSI of 200 bytes adds 1,000. If a query needs two attributes, `INCLUDE` those two and leave large blobs out of the index entirely. Common mistake: projecting `ALL` because it makes one query convenient, then paying for it on every write, or using `KEYS_ONLY` and turning one query into N follow-up `GetItem` calls.

In [ ]:
def ceil_div(numerator, denominator):
    return -(-numerator // denominator)


def write_units(item_bytes, writes_per_second):
    return ceil_div(item_bytes, 1024) * writes_per_second


writes = 1_000
base = write_units(1500, writes)
with_all = base + write_units(1500, writes)
with_keys_only = base + write_units(200, writes)

print(f"base = {base} WCU, +ALL index = {with_all}, +KEYS_ONLY index = {with_keys_only}")
assert base == 2000 and with_all == 4000 and with_keys_only == 3000


## How do you use conditional writes and optimistic locking?

Conditional writes make an operation atomic and fail with `ConditionalCheckFailedException` when a condition is not met. They are how DynamoDB implements uniqueness, state machines, and optimistic locking without a server-side transaction.

Enforce uniqueness and valid state transitions:

In [ ]:
# ILLUSTRATIVE - requires boto3, botocore: install to run, or read on.
import boto3
from botocore.exceptions import ClientError
from boto3.dynamodb.conditions import Attr

table = boto3.resource("dynamodb", region_name="us-east-1").Table("orders")

try:
    table.put_item(
        Item={"PK": "CUSTOMER#123", "SK": "ORDER#2026-10-04#999", "status": "created"},
        ConditionExpression="attribute_not_exists(PK)",
    )
except ClientError as exc:
    if exc.response["Error"]["Code"] == "ConditionalCheckFailedException":
        raise ValueError("order already exists") from exc
    raise

table.update_item(
    Key={"PK": "CUSTOMER#123", "SK": "ORDER#2026-10-04#999"},
    UpdateExpression="SET #s = :new, updated_at = :ts",
    ConditionExpression="#s = :expected",
    ExpressionAttributeNames={"#s": "status"},
    ExpressionAttributeValues={":new": "shipped", ":expected": "created", ":ts": "2026-10-04T13:00:00Z"},
)

Optimistic locking with a version attribute is the standard pattern for read-modify-write:

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
def update_with_retry(key: dict, mutate, retries: int = 5) -> None:
    for _ in range(retries):
        item = table.get_item(Key=key, ConsistentRead=True)["Item"]
        new_item = mutate(item)
        new_item["version"] = int(item.get("version", 0)) + 1
        try:
            table.put_item(
                Item=new_item,
                ConditionExpression="version = :v",
                ExpressionAttributeValues={":v": int(item.get("version", 0))},
            )
            return
        except ClientError as exc:
            if exc.response["Error"]["Code"] != "ConditionalCheckFailedException":
                raise
    raise RuntimeError("too much contention on item")

Notes:

+ A failed condition still consumes write capacity.
+ Conditions are evaluated atomically against the current item, so they work across concurrent writers without external locks.
+ Do not build distributed locks out of conditional writes unless you own the lease renewal and expiry semantics; a crashed holder can block progress.

How a condition works: the condition is evaluated against the stored item inside the same atomic operation as the write, so no other writer can interleave. `attribute_not_exists` enforces create-if-absent, an equality on a version attribute enforces optimistic locking, and a comparison on a state attribute enforces legal transitions. A failed condition raises `ConditionalCheckFailedException` and still consumes the write capacity of the request.

Worked example: read the item with `ConsistentRead=True`, compute the next version, then `PutItem` with `ConditionExpression="version = :v"`. If another writer won the race the condition fails, so re-read and retry with jitter. This is compare-and-swap: it serialises only on the contended item, not on the table. Common mistake: retrying a failed condition in a tight loop, which turns contention into a hot partition, or expecting `attribute_not_exists(PK)` to enforce uniqueness on some other attribute when the table key is `PK` plus `SK`. If uniqueness must span attributes, put the unique value in a key or write a dedicated lock item.

## How do you use DynamoDB transactions?

`TransactWriteItems` and `TransactGetItems` give all-or-nothing atomicity across up to 100 items, which can span multiple tables in the same account and Region. They are the escape hatch for invariants that cannot be modeled as a single item, such as debiting one account and crediting another.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
client = boto3.client("dynamodb", region_name="us-east-1")

client.transact_write_items(
    TransactItems=[
        {
            "Update": {
                "TableName": "ledger",
                "Key": {"PK": {"S": "ACCOUNT#A"}, "SK": {"S": "BALANCE"}},
                "UpdateExpression": "SET balance = balance - :amt",
                "ConditionExpression": "balance >= :amt",
                "ExpressionAttributeValues": {":amt": {"N": "100.00"}},
            }
        },
        {
            "Update": {
                "TableName": "ledger",
                "Key": {"PK": {"S": "ACCOUNT#B"}, "SK": {"S": "BALANCE"}},
                "UpdateExpression": "SET balance = balance + :amt",
                "ExpressionAttributeValues": {":amt": {"N": "100.00"}},
            }
        },
        {
            "Put": {
                "TableName": "ledger",
                "Item": {
                    "PK": {"S": "TXN#20261004#0001"},
                    "SK": {"S": "LEDGER"},
                    "amount": {"N": "100.00"},
                },
                "ConditionExpression": "attribute_not_exists(PK)",
            }
        },
    ],
    ClientRequestToken="txn-20261004-0001",
)

Constraints and failure modes:

+ 100 items maximum, 4 MB total, no two operations on the same item in one transaction.
+ Transactions are not supported across Regions or accounts.
+ You cannot run a `Query` inside a transaction; you must know the keys in advance. This pushes reads and business decisions before the transaction, which is when conflicts appear.
+ `TransactionConflictException` means another transaction touched the same item; retry with backoff.
+ Cost is higher: transactional writes roughly double the WCU, and `TransactWriteItems` cannot target a GSI directly.
+ Prefer single-item conditional updates when they suffice; reach for transactions only when the invariant genuinely spans items.

How it works: `TransactWriteItems` groups up to 100 actions across tables, checks every condition, then commits or cancels the whole set. `TransactGetItems` reads up to 100 items as one consistent snapshot. Transactions cost more than single writes because DynamoDB prepares and then commits, and a `TransactionConflictException` means another transaction touched the same item, so the client retries the whole transaction with backoff.

The constraints that shape design: 100 items and 4 MB per transaction, no two actions on the same item, no `Query` inside a transaction, no cross-Region or cross-account transaction, and no direct write to a GSI because the base write propagates. `ClientRequestToken` makes a transaction idempotent for a short window, so a retried network call does not post twice. Common mistake: using transactions as a batching mechanism for unrelated writes. Every item in a transaction shares a fate, and unrelated items on different partitions create conflicts for no benefit. Reserve transactions for invariants that genuinely span items, such as a debit and a credit.

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
# Requires boto3 and AWS credentials; shown for review.
import boto3

client = boto3.client("dynamodb", region_name="us-east-1")

# One consistent snapshot of two related items.
response = client.transact_get_items(
    TransactItems=[
        {"Get": {"TableName": "ledger",
                 "Key": {"PK": {"S": "ACCOUNT#A"}, "SK": {"S": "BALANCE"}}}},
        {"Get": {"TableName": "ledger",
                 "Key": {"PK": {"S": "ACCOUNT#B"}, "SK": {"S": "BALANCE"}}}},
    ]
)
balances = [item["Item"] for item in response["Responses"]]


## How do you capture changes with DynamoDB Streams and TTL?

DynamoDB Streams records an ordered log of item-level changes (with `KEYS_ONLY`, `NEW_IMAGE`, `OLD_IMAGE`, or `NEW_AND_OLD_IMAGES`) and retains it for 24 hours. It is the standard way to trigger side effects, replicate to another store, and build CDC pipelines into a lakehouse.

```bash
aws dynamodb update-table --table-name orders \
  --stream-specification StreamEnabled=true,StreamViewType=NEW_AND_OLD_IMAGES
```

Lambda consumer that forwards to Kinesis Firehose for lake landing:

In [ ]:
# ILLUSTRATIVE - requires boto3: install to run, or read on.
import json
import boto3
from decimal import Decimal

firehose = boto3.client("firehose")

def handler(event, context):
    records = []
    for record in event["Records"]:
        if record["eventName"] not in {"INSERT", "MODIFY", "REMOVE"}:
            continue
        image = record["dynamodb"].get("NewImage") or record["dynamodb"].get("OldImage")
        payload = {
            "event_name": record["eventName"],
            "event_id": record["eventID"],
            "keys": record["dynamodb"]["Keys"],
            "image": image,
        }
        records.append({
            "Data": json.dumps(payload, default=lambda o: float(o) if isinstance(o, Decimal) else str(o))
        })
    if records:
        firehose.put_record_batch(DeliveryStreamName="orders-cdc", Records=records)
    return {"forwarded": len(records)}

TTL deletes items automatically by comparing a numeric epoch-seconds attribute against the current time:

```bash
aws dynamodb update-time-to-live --table-name sessions \
  --time-to-live-specification "Enabled=true,AttributeName=expires_at"
```

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
import time
table.put_item(Item={
    "PK": "SESSION#abc",
    "SK": "META",
    "user_id": "123",
    "expires_at": int(time.time()) + 3600,
})

Notes:

+ Streams are ordered per partition key, not globally. Downstream consumers must be idempotent and tolerate at-least-once delivery.
+ TTL deletion is free but not immediate; items can linger for up to ~48 hours and the delete appears in the stream with `eventName=REMOVE` and `userIdentity.principalId=dynamodb.amazonaws.com`. Do not rely on TTL for strict data-retention guarantees.
+ `NEW_AND_OLD_IMAGES` costs more and can double stream payload size; choose the smallest view type the consumer needs.
+ For lower latency and fan-out beyond Lambda's concurrency, send the stream to Kinesis Data Streams for DynamoDB and read from there.

How Streams works: each change is written to a shard associated with the item's partition, so ordering is guaranteed per partition key but not across the table. The stream retains records for 24 hours, and the view type (`KEYS_ONLY`, `NEW_IMAGE`, `OLD_IMAGE`, `NEW_AND_OLD_IMAGES`) decides how much of the item each record carries. Delivery is at-least-once, so consumers must be idempotent; the record's sequence number or the item's natural key is the usual dedup key.

How TTL works: you nominate one numeric attribute as the TTL attribute and store epoch seconds in it. DynamoDB compares it with the current time and deletes the item in the background, typically within 48 hours of expiry, at no write cost, recording the delete in the stream as a `REMOVE` with `userIdentity.principalId = dynamodb.amazonaws.com`. A read can still return an item whose TTL has passed but which has not yet been deleted, so filter expired items in the client when that matters.

Worked example: 500 sessions/s with a 7-day TTL reaches a steady state of 302,400,000 items, about 577 GiB at 2 KiB each. The stream carries an INSERT and later a REMOVE per session, about 86.4 million records per day. Because the stream retains only 24 hours, a consumer that can be down longer than that needs a sink such as Firehose to S3, or Kinesis with longer retention, plus a replay path.

In [ ]:
def steady_state_items(creates_per_second, ttl_days):
    return creates_per_second * ttl_days * 86_400


sessions = steady_state_items(500, 7)
print(f"steady-state sessions = {sessions:,}")
print(f"at 2 KiB each = {sessions * 2048 / 1024 ** 3:.1f} GiB")
assert sessions == 302_400_000


Inspect a stream and its TTL configuration:

```bash
aws dynamodb describe-table --table-name orders \
  --query 'Table.{StreamArn:LatestStreamArn,StreamView:StreamSpecification}'

aws dynamodb describe-time-to-live --table-name sessions

# Read a stream shard with the low-level API.
aws dynamodbstreams get-shard-iterator \
  --stream-arn arn:aws:dynamodb:us-east-1:123456789012:table/orders/stream/2026-10-04T00:00:00.000 \
  --shard-id shardId-00000000000000000000 \
  --shard-iterator-type LATEST
```


## How do you model one-to-many and many-to-many relationships?

DynamoDB has no joins, so relationships are modeled by duplicating data and choosing keys that cluster related items. The two core techniques are item collections (one partition key, many sort keys) and the adjacency-list pattern (each relationship becomes an item, queryable from both directions).

One-to-many, for example a customer with orders and each order with line items:

```text
PK = CUSTOMER#123   SK = ORDER#2026-10-04#999
PK = ORDER#999      SK = ITEM#1
PK = ORDER#999      SK = ITEM#2
```

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
# All line items for an order
table.query(
    KeyConditionExpression=Key("PK").eq("ORDER#999") & Key("SK").begins_with("ITEM#")
)

Many-to-many with an adjacency list, for example users and projects with memberships:

```text
PK = USER#u1    SK = PROJECT#p1   role=admin
PK = PROJECT#p1 SK = USER#u1      role=admin
```

Writing both items in one `TransactWriteItems` keeps the relationship consistent, and each direction is a single `Query`. This doubles write cost per relationship but makes both traversals O(1) partitions.

Design rules that interviewers look for:

+ Duplicate the attributes needed by the query so you never need a join; denormalization is expected, and you accept update fan-out as the cost.
+ Use `begins_with` on hierarchical sort keys to model trees, for example `SK = PATH#/a/b/c`.
+ Keep item collections bounded. A single hot partition key cannot exceed one partition's throughput, and LSIs add a 10 GB per-key ceiling.
+ For high-fan-out aggregates (follower counts, likes), use sharded counters and sum on read rather than one hot item.
+ Version or soft-delete related items rather than performing multi-item deletes; then compact them with a background job.

How the patterns work: an item collection clusters related rows under one partition key, and an adjacency list adds a second item for the reverse direction so both traversals are a `Query`. A composite sort key such as `ORDER#2026-10-04#999` sorts by several dimensions at once, which is how one index answers date-range and status queries without extra tables.

Worked example: `PK = USER#u1, SK = PROJECT#p1` and `PK = PROJECT#p1, SK = USER#u1` are written in one transaction. Both "list a user's projects" and "list a project's users" are single-partition queries, and duplicated attributes mean neither needs a follow-up read. The cost is two writes per relationship and the risk of the items drifting unless they are always written together. Common mistake: letting one partition key grow without bound. An item collection with an LSI is capped at 10 GB, and any collection is capped by the throughput of one partition, so high-fan-out relationships need sharding or a GSI keyed by a higher-cardinality attribute. A second mistake is storing the reverse edge lazily, which makes one traversal fast and the other a `Scan`.

The two adjacency items carry the same membership in both directions:

```json
[
  {"PK": {"S": "USER#u1"}, "SK": {"S": "PROJECT#p1"},
   "entity_type": {"S": "membership"}, "role": {"S": "admin"},
   "GSI1PK": {"S": "PROJECT#p1"}, "GSI1SK": {"S": "USER#u1"}},
  {"PK": {"S": "PROJECT#p1"}, "SK": {"S": "USER#u1"},
   "entity_type": {"S": "membership"}, "role": {"S": "admin"},
   "GSI1PK": {"S": "USER#u1"}, "GSI1SK": {"S": "PROJECT#p1"}}
]
```


## How do you handle throttling, retries, and errors?

The errors you must recognize:

+ `ProvisionedThroughputExceededException` / `ThrottlingException`: capacity or partition-level limit exceeded. Retry with exponential backoff and jitter.
+ `ConditionalCheckFailedException`: a business condition failed; usually do not retry blindly.
+ `TransactionConflictException`: concurrent transaction on the same item; retry the whole transaction.
+ `ItemCollectionSizeLimitExceededException`: an LSI item collection exceeded 10 GB.
+ `ValidationException`: malformed expression, reserved word, or wrong key type.
+ `RequestLimitExceeded`: too many control-plane calls; back off.

Retry with full jitter using boto3's adaptive mode when appropriate:

In [ ]:
# ILLUSTRATIVE - requires boto3, botocore: install to run, or read on.
from botocore.config import Config
import boto3

config = Config(
    retries={"max_attempts": 10, "mode": "adaptive"},
    max_pool_connections=50,
)
client = boto3.client("dynamodb", region_name="us-east-1", config=config)

In [ ]:
# ILLUSTRATIVE - requires botocore: install to run, or read on.
import random
import time
from botocore.exceptions import ClientError

RETRYABLE = {"ProvisionedThroughputExceededException", "ThrottlingException", "RequestLimitExceeded"}

def with_backoff(fn, attempts: int = 8):
    for attempt in range(attempts):
        try:
            return fn()
        except ClientError as exc:
            code = exc.response["Error"]["Code"]
            if code not in RETRYABLE or attempt == attempts - 1:
                raise
            time.sleep(random.uniform(0, min(2 ** attempt, 20)) * 0.1)

Operational guidance:

+ Prefer `BatchWriteItem` (25 items) and `BatchGetItem` (100 keys) for throughput, but remember batch calls can partially fail and return `UnprocessedItems`; loop until empty with backoff.
+ Distinguish retryable throttling from a modeling problem. Persistent throttling on one key means the key is hot, not that retries are too few.
+ Add jitter; synchronized retries create thundering herds.
+ Track throttles per table and per index, and use Contributor Insights for per-key visibility.
+ Make consumers idempotent, because at-least-once delivery from Streams, Lambdas, and retries will replay operations.

What throttling means: `ProvisionedThroughputExceededException` and `ThrottlingException` mean the request exceeded either table or index capacity or a single partition's limit. `RequestLimitExceeded` means too many control-plane calls. These are retryable with exponential backoff and jitter. `ValidationException` and `ConditionalCheckFailedException` are not, because retrying repeats a deterministic failure.

What to read in CloudWatch: `ThrottledRequests` counts throttled requests by table and index, `ReadThrottleEvents` and `WriteThrottleEvents` show which direction they came from, `ConsumedReadCapacityUnits` and `ConsumedWriteCapacityUnits` show how close to provisioned you run, and `UserErrors` separates client mistakes from capacity. `ReturnedItemCount` and `SuccessfulRequestLatency` show whether queries return more than expected. Contributor Insights reports the most accessed partition keys, which is the fastest way to find a hot key. Sustained throttle events with consumed units near provisioned means add capacity or fix a hot key; throttles with low consumed units point at one key hitting the 1,000 WCU partition ceiling.

Common mistake: reading `ThrottledRequests` alone, which mixes table and index and hides which one is short; increasing retries without jitter, which synchronises every client into a thundering herd; or raising provisioned capacity when a single hot key is the problem. The SDK's adaptive retry mode adds client-side rate limiting on top of backoff.

In [ ]:
RETRYABLE = {
    "ProvisionedThroughputExceededException",
    "ThrottlingException",
    "RequestLimitExceeded",
    "TransactionConflictException",
    "InternalServerError",
}


def classify(code):
    return "retry with backoff" if code in RETRYABLE else "do not retry"


for error_code in ["ThrottlingException", "ConditionalCheckFailedException",
                   "TransactionConflictException", "ValidationException"]:
    print(f"{error_code:<38} -> {classify(error_code)}")

assert classify("ConditionalCheckFailedException") == "do not retry"
assert classify("TransactionConflictException") == "retry with backoff"


Pull the throttle metrics rather than guessing:

```bash
aws cloudwatch get-metric-statistics \
  --namespace AWS/DynamoDB \
  --metric-name ThrottledRequests \
  --dimensions Name=TableName,Value=orders \
  --start-time 2026-10-04T00:00:00Z --end-time 2026-10-04T06:00:00Z \
  --period 300 --statistics Sum

aws cloudwatch get-metric-statistics \
  --namespace AWS/DynamoDB \
  --metric-name ConsumedWriteCapacityUnits \
  --dimensions Name=TableName,Value=orders \
  --start-time 2026-10-04T00:00:00Z --end-time 2026-10-04T06:00:00Z \
  --period 300 --statistics Sum
```


## When should you NOT use DynamoDB?

DynamoDB is excellent at predictable, single-digit-millisecond key-based access at very large scale. It is a poor fit when the query shape is unknown or the data is inherently relational.

Avoid DynamoDB when:

+ You need ad-hoc analytics, joins, or arbitrary `GROUP BY` aggregations. Use a warehouse or an OLAP store, and land DynamoDB data there via Streams or export to S3.
+ Query patterns are still changing weekly. Every new pattern can require a new GSI and a backfill, and index key schemas are immutable.
+ The workload is read-heavy with large scans over a modest table. A relational database with an index will be cheaper and simpler.
+ You need multi-item ACID transactions beyond 100 items, or cross-Region transactions.
+ Strong consistency across the whole dataset is required without design work; DynamoDB is per-item consistent, and GSIs are always eventually consistent.
+ Long, complex queries with server-side aggregation, window functions, or `ORDER BY` over non-key columns are core requirements.
+ The team needs a SQL-first workflow with BI tools pointing directly at the database.

Alternatives and when they fit:

+ Amazon Aurora/PostgreSQL for relational integrity, joins, and moderate scale.
+ Amazon Bigtable or Cassandra for wide-column, high-write time-series workloads with a different operational model.
+ Amazon Redshift, Athena, or an open lakehouse for analytics on the same data.
+ Elasticsearch/OpenSearch for full-text search and flexible filtering.

Interview answer: choose DynamoDB when access patterns are known, keys can be designed for even distribution, and the dominant need is horizontal scale at low latency. Otherwise name the access pattern that breaks it and propose the store that fits.

The decision is about access patterns and guarantees, not about scale. DynamoDB gives per-item atomicity, single-digit-millisecond key access, and horizontal scale, but it gives up joins, ad-hoc predicates, and cross-item consistency outside transactions. If the product needs any of those as a first-class feature, another store is usually cheaper and simpler than forcing DynamoDB to imitate a relational database.

Common mistake: choosing DynamoDB for a system whose queries are still being discovered. Every new access pattern is a new GSI and a backfill, and index key schemas are immutable, so the cost of being wrong is a migration. A second mistake is assuming a GSI can serve strongly consistent reads; it cannot, so a read-your-writes requirement must be designed onto the base table.

## How do you design keys for even access and avoid hot partitions?

Even access is a property of the key, not of the table. A partition key is good when its values are numerous, roughly uniform in load, and known by the queries that need them. Trouble comes from a small set of values, from one value that grows without bound, and from monotonic keys such as a timestamp or an auto-increment id that send every new write to the same place.

The fixes, in order of preference. First, choose a naturally high-cardinality key from the query itself, such as `device_id` or `order_id`. Second, widen the key, for example `PK = CUSTOMER#123` for an item collection, or `PK = CUSTOMER#123#STATUS#shipped` so a low-cardinality status is spread across customers. Third, shard an irreducible hot key with a deterministic suffix `#BUCKET#<hash>`; reads that know the attribute compute the bucket, and reads that need all shards fan out in parallel and merge.

Worked example: 25,000 writes/s to one logical `STATUS#active` key. One partition absorbs about 1,000 WCU, so the key throttles. 25 buckets would just fit and 32 leaves headroom; the application computes `crc32(id) % 32` and writes to `STATUS#active#<bucket>`. A read for a specific id touches one bucket; the aggregate read issues 32 `Query` calls and sums. Common mistake: using random buckets for a key that queries address by identity, which forces fan-out on every read, and salting without changing the read path, which just moves the scan.

In [ ]:
def buckets_needed(peak_wcu, partition_wcu=1_000):
    return -(-peak_wcu // partition_wcu)


for peak in (800, 4_000, 25_000):
    needed = buckets_needed(peak)
    print(f"peak {peak:>6} WCU -> {needed:>2} buckets of {peak / needed:7.1f} WCU")

assert buckets_needed(1_000) == 1
assert buckets_needed(25_000) == 25


## How do you plan capacity with WCU and RCU maths?

Start from the peak second and the p99 item size, never the averages. Convert traffic to units: writes are `ceil(item_bytes / 1024)` WCU per write per second; strongly consistent reads are `ceil(item_bytes / 4096)` RCU per read per second, halved for eventually consistent. Add the fan-out: every GSI that projects a changed attribute adds its own units, and a transactional write costs roughly double.

Then choose a mode. Provisioned with Application Auto Scaling is cheaper above roughly 15 percent average utilisation of the provisioned floor, because you pay for reserved capacity per hour while on-demand bills per request. On-demand is cheaper below that and for spiky or unknown traffic, and it needs no scaling policy. Autoscaling targets a utilisation such as 70 percent, scales out in response to consumed capacity, and scales in slowly to avoid flapping.

Worked example: with us-east-1 standard-table list prices of $0.00065 per WCU-hour and $1.25 per million write request units, one provisioned WCU costs $0.00065 per hour, while the same write per second sustained for a full hour on demand costs $0.0045, so break-even is about 14.4 percent utilisation. Re-check current prices before quoting them, because they vary by Region. Common mistake: provisioning for the peak and paying for it all day, or provisioning for the average and throttling every peak. Autoscaling narrows that gap but reacts in minutes, so keep a burst margin, and remember that each GSI has its own capacity setting.

In [ ]:
PROVISIONED_WCU_HOUR = 0.00065
ON_DEMAND_WCU_MILLION = 1.25


def provisioned_cost(wcu, hours):
    return wcu * PROVISIONED_WCU_HOUR * hours


def on_demand_cost(writes):
    return writes / 1_000_000 * ON_DEMAND_WCU_MILLION


def break_even_utilisation(hours):
    return provisioned_cost(1, hours) / on_demand_cost(hours * 3600)


print(f"break-even utilisation = {break_even_utilisation(1):.1%}")
assert round(break_even_utilisation(1), 3) == 0.144


## How do you reduce cost and latency in DynamoDB?

DAX is an in-memory cache in front of the table that serves eventually consistent reads in microseconds and absorbs repeated reads so they never consume RCU. It is write-through, so a write updates the cache and the table; it does not make reads strongly consistent and it does not help write-heavy or scan-heavy access. An item-cache hit returns a point read, and a query-cache hit returns a repeated query.

The structural fixes usually matter more. Batch 25 writes or 100 reads per call to cut round trips. Project only the attributes a hot query needs, and use `KEYS_ONLY` plus a follow-up read only when that extra read is rare. Make a GSI sparse so it indexes only the small subset of items a query cares about. Turn on TTL to expire data for free instead of deleting and storing it. Keep large blobs in S3 and store a pointer. Common mistake: caching a write-dominated workload, expecting DAX to serve a strongly consistent read, or using `KEYS_ONLY` on a hot query and turning one query into 100 follow-up reads.

Worked example: 8,000 reads/s of a 4 KB item costs 4,000 RCU/s eventually consistent. With a 65 percent item-cache hit and a 20 percent query-cache hit, only 15 percent of reads reach the table, cutting RCU to 600/s, an 85 percent reduction. The DAX cluster has its own hourly cost, so the saving only pays when read volume is high enough.

In [ ]:
def rcu_per_read(item_kb):
    # DAX serves eventually consistent reads only.
    return -(-item_kb // 4) / 2


reads_per_second = 8_000
missing = 1 - 0.65 - 0.20

without_dax = reads_per_second * rcu_per_read(4)
with_dax = reads_per_second * missing * rcu_per_read(4)
saved_pct = (without_dax - with_dax) / without_dax

print(f"without DAX = {without_dax:,.0f} RCU/s")
print(f"with DAX    = {with_dax:,.0f} RCU/s (miss {missing:.0%})")
print(f"saved       = {saved_pct:.0%}")
assert without_dax == 4000 and round(with_dax, 6) == 600
assert round(saved_pct, 2) == 0.85


## How do you diagnose throttling and errors?

Start from the metric that matches the symptom. `ThrottledRequests` tells you how often requests were rejected; `ReadThrottleEvents` and `WriteThrottleEvents` tell you which direction; `ConsumedReadCapacityUnits` and `ConsumedWriteCapacityUnits` against `ProvisionedReadCapacityUnits` and `ProvisionedWriteCapacityUnits` tell you whether the table is near its ceiling. High consumption with throttling means the table is short of capacity; throttling with low consumption means one partition is hot.

Split client errors from capacity errors with `UserErrors`, and watch index-level metrics separately, because a GSI has its own throttle events and consumed units. Contributor Insights reports the top partition keys by access count and throttled requests, which answers "which key" directly. Without it, compare `ReturnedItemCount` and latency by operation to see whether one access pattern dominates.

The fix depends on the cause. Capacity shortfall: raise provisioned units or move to on-demand, and scale the GSI separately. Hot key: shard it, cache it, or change the access pattern. Purely bursty traffic: rely on burst capacity and jittered retries. `TransactionConflictException` is contention, not capacity, so back off the whole transaction; `ConditionalCheckFailedException` is a business outcome, so do not retry it.

In [ ]:
import random


def full_jitter_delays(attempts, base=0.05, cap=2.0, seed=7):
    rng = random.Random(seed)
    return [round(rng.uniform(0, min(cap, base * 2 ** i)), 4) for i in range(attempts)]


delays = full_jitter_delays(6)
print("full-jitter delays:", delays)
assert len(delays) == 6
assert all(0 <= delay <= 2.0 for delay in delays)


The retry pattern in application code keeps the delay random but the total bounded:

```python
delay = min(cap, base * 2 ** attempt)
sleep(random.uniform(0, delay))   # full jitter, not the full delay
```


## Real world use cases for DynamoDB

The cases below are worked end to end: the access pattern, the key design, the mechanism that makes it fast, and the operational caveat. Read each one as a template. The question to ask is always which single partition a hot request touches, and whether any key can exceed 3,000 RCU or 1,000 WCU.

### 1. E-commerce cart and order state (multi-tenant SaaS)

Business context: a marketplace with 8M monthly active buyers needs a cart that survives devices, plus an order state machine. Volume: ~120M cart writes/day at peak 15k writes/s, items under 2 KB.

```text
PK = CART#<tenant>#<user_id>   SK = ITEM#<sku>
PK = ORDER#<order_id>          SK = META
GSI1: GSI1PK = USER#<user_id>#STATUS#<status>, GSI1SK = order_ts
```

`PutItem` with `ConditionExpression` guards item quantity, and `UpdateItem` transitions order status with a condition on the allowed previous state. `cart` uses TTL to expire abandoned carts after 30 days.

Why this design: every hot access pattern is a single-partition `Query`, conditional writes make concurrent device updates safe without a lock, and TTL removes abandoned-cart storage cost automatically. Tenant is part of the partition key so a large tenant cannot starve the others.

### 2. Fintech ledger and idempotent payments

Business context: a payments API must never double-post a transfer and must expose an audit trail. Volume: ~5M transfers/day, ledger item ~1 KB, strict p99 under 20 ms.

In [ ]:
# ILLUSTRATIVE - needs the live engine session and data this section names; read on for the pattern.
client.transact_write_items(
    TransactItems=[
        {"Update": {"TableName": "ledger",
                    "Key": {"PK": {"S": "ACCOUNT#A"}, "SK": {"S": "BALANCE"}},
                    "UpdateExpression": "SET balance = balance - :amt",
                    "ConditionExpression": "balance >= :amt",
                    "ExpressionAttributeValues": {":amt": {"N": "250.00"}}}},
        {"Put": {"TableName": "ledger",
                 "Item": {"PK": {"S": "IDEMPOTENCY#req-7f3a"}, "SK": {"S": "TX"},
                          "response": {"S": "accepted"}},
                 "ConditionExpression": "attribute_not_exists(PK)"}},
    ],
    ClientRequestToken="req-7f3a",
)

Why this design: the delete-and-credit pair is atomic, and the idempotency item makes retries safe. The same `ClientRequestToken` lets DynamoDB reject duplicate submissions. Streams feed both the fraud pipeline and a lakehouse audit table.

### 3. IoT telemetry latest-state and alerting

Business context: 250k devices report every 10 seconds; the app needs the latest reading per device and hot-device alerting, while the full time series lives in S3/Timestream. Volume: 2.2B writes/day, ~1 KB per reading.

```text
PK = DEVICE#<device_id>   SK = LATEST
GSI1: GSI1PK = SITE#<site_id>#STATE#<state>, GSI1SK = last_seen
```

Writes overwrite `LATEST` with a conditional `last_seen > :prev` so out-of-order retries cannot regress state. A sparse GSI holds only devices currently in `ALARM`, so alert queries are tiny.

Why this design: overwriting a single item per device keeps storage flat, the sparse index keeps alerting cheap, and the bulk time series is streamed to cheap object storage rather than stored in DynamoDB.

### 4. CDC from DynamoDB into an open lakehouse

Business context: the analytics team needs near-real-time order facts in Iceberg without querying the production table. Volume: 120M change events/day, peak 5 MB/s.

```text
DynamoDB Streams (NEW_AND_OLD_IMAGES)
  -> Lambda (batch 500 records, idempotent by eventID)
  -> Kinesis Firehose (60s buffer)
  -> s3://lake/bronze/dynamodb_orders/  (JSON, partitioned by ingest date)
  -> Spark Structured Streaming MERGE INTO iceberg.analytics.orders
```

Why this design: Streams is push-based and adds no read load to the serving table, Firehose buffers and batches to avoid small files, and the `eventID` gives the lakehouse a natural dedup key for the Spark merge.

### 5. Gaming leaderboard and session store

Business context: 3M daily players need sub-10 ms leaderboard reads and session lookups that expire. Volume: ~50k updates/s at peak during events.

```text
PK = LEADERBOARD#<board_id>   SK = SCORE#<padded_score>#<player_id>
PK = SESSION#<session_id>     SK = META     (TTL expires_at)
```

Scores are zero-padded so `ScanIndexForward=False` returns the top N in score order without sorting server-side. DAX sits in front to absorb repeated leaderboard reads.

Why this design: `Query` with a descending sort on the sort key returns the top players without aggregation, DAX cuts read cost on the hot board, and TTL handles sessions without a cleanup job.

## DynamoDB exercises

These exercises use only the Python standard library and model the capacity and key-design ideas with small, runnable calculations. Each one names the task, gives a hint, states the expected result, and then prints or asserts the answer. Run them with the repository virtual environment: `/Users/sai/WS/data-engineering-interview-questions/.venv/bin/python`.

The exercises use the same units DynamoDB bills in. Writes are `ceil(item_bytes / 1024)` WCU per write per second, strongly consistent reads are `ceil(item_bytes / 4096)` RCU per read per second and eventually consistent reads cost half, and a single partition serves at most 3,000 RCU or 1,000 WCU. The item-collection exercise uses the 10 GB ceiling that applies when a table has one or more local secondary indexes.

Treat the asserted numbers as checks on the formula rather than predictions of a real table. Real runs add variance from caching, adaptive capacity, and traffic skew, so the value of each exercise is knowing which quantity to compute and which direction a change moves it.

### Try it 1. Compute RCU and WCU from item size and consistency

**Task.** A 3.5 KB item receives 1,200 writes/s and 400 reads/s. Compute the WCU, the RCU for strongly consistent reads, and the RCU for eventually consistent reads.

**Hint.** Writes round up per KB, reads round up per 4 KB, and eventually consistent reads cost half of a strongly consistent read.

**Expected result.** `4,800` WCU, `400` RCU strongly consistent, `200` RCU eventually consistent.

In [ ]:
def ceil_div(numerator, denominator):
    return -(-numerator // denominator)


def write_units(item_kb, writes_per_second):
    return ceil_div(item_kb, 1) * writes_per_second


def read_units(item_kb, reads_per_second, strongly_consistent=True):
    units = ceil_div(item_kb, 4)
    return units * reads_per_second if strongly_consistent else units * reads_per_second / 2


wcu = write_units(3.5, 1_200)
strong = read_units(3.5, 400, True)
eventual = read_units(3.5, 400, False)
print(f"3.5 KB x 1200 writes/s   = {wcu:,.0f} WCU")
print(f"3.5 KB x 400 strong/s    = {strong:,.0f} RCU")
print(f"3.5 KB x 400 eventual/s  = {eventual:,.0f} RCU")
assert wcu == 4800 and strong == 400 and eventual == 200


### Try it 2. Decide on-demand versus provisioned from a traffic profile

**Task.** A flash sale drives 10,000 writes/s of 1 KB for one hour a day; the rest of the day is near idle. Using us-east-1 standard-table list prices of $0.00065 per WCU-hour and $1.25 per million write request units, compute the daily cost of provisioning 10,000 WCU for 24 hours and of paying on demand for one hour, then choose the cheaper mode.

**Hint.** Provisioned cost is `wcu * price_per_wcu_hour * 24`; on-demand cost is `wcu * hours * 3600 / 1_000_000 * price_per_million`.

**Expected result.** On-demand is about `$45.00` per day and provisioned about `$156.00`, so on-demand wins.

In [ ]:
PROVISIONED_WCU_HOUR = 0.00065
ON_DEMAND_WCU_MILLION = 1.25
peak_wcu = 10_000
peak_hours = 1


def provisioned_daily(wcu):
    return wcu * PROVISIONED_WCU_HOUR * 24


def on_demand_daily(wcu, hours):
    return wcu * hours * 3600 / 1_000_000 * ON_DEMAND_WCU_MILLION


on_demand = on_demand_daily(peak_wcu, peak_hours)
provisioned = provisioned_daily(peak_wcu)
choice = "on-demand" if on_demand < provisioned else "provisioned"
print(f"on-demand  = ${on_demand:.2f}/day")
print(f"provisioned = ${provisioned:.2f}/day")
print("choice:", choice)
assert round(on_demand, 2) == 45.00
assert round(provisioned, 2) == 156.00
assert choice == "on-demand"


### Try it 3. Detect a hot partition from key counts

**Task.** Over a 10-second window a workload keyed by `DATE#2026-10-04` receives 30,000 1 KB writes, and four other keys receive 4,000, 3,000, 2,000, and 1,000. Convert to writes per second, flag every key above the 1,000 WCU partition limit, and compute how many shard buckets the hottest key needs.

**Hint.** Divide each count by the window length, compare with 1,000, then compute `ceil(hottest_rate / 1_000)`.

**Expected result.** Only `DATE#2026-10-04` is hot at `3,000.0` WCU/s, and it needs `3` buckets.

In [ ]:
PARTITION_WCU = 1_000
WINDOW_SECONDS = 10
counts = {
    "DATE#2026-10-04": 30_000,
    "USER#1": 4_000,
    "USER#2": 3_000,
    "USER#3": 2_000,
    "TENANT#acme": 1_000,
}

rates = {key: count / WINDOW_SECONDS for key, count in counts.items()}
hot = [key for key, rate in rates.items() if rate > PARTITION_WCU]
for key, rate in sorted(rates.items(), key=lambda kv: -kv[1]):
    flag = "HOT" if rate > PARTITION_WCU else ""
    print(f"{key:<18} {rate:7.1f} WCU/s {flag}")

hottest = max(rates.values())
buckets = -(-int(hottest) // PARTITION_WCU)
print("hot keys:", hot)
print("shard buckets needed:", buckets)
assert hot == ["DATE#2026-10-04"]
assert buckets == 3


### Try it 4. Size an item collection against the 10 GB limit

**Task.** A table with an LSI holds 12,000,000 items of 1 KiB under a single partition key. Compute the collection size in GiB, decide whether it exceeds the 10 GiB limit, and compute the maximum item count at this size.

**Hint.** One GiB is `1024 ** 3` bytes, and the limit is 10 GiB. Divide the limit by the item size to get the maximum count.

**Expected result.** About `11.44` GiB, so it exceeds the limit, and the ceiling is `10,485,760` items.

In [ ]:
GiB = 1024 ** 3
LSI_COLLECTION_LIMIT = 10 * GiB

items = 12_000_000
item_bytes = 1024
total_bytes = items * item_bytes
total_gib = total_bytes / GiB
over_limit = total_bytes > LSI_COLLECTION_LIMIT
max_items = LSI_COLLECTION_LIMIT // item_bytes

print(f"{items:,} items x 1 KiB = {total_gib:.2f} GiB")
print("exceeds 10 GiB LSI limit:", over_limit)
print(f"max items at 1 KiB = {max_items:,}")
assert over_limit is True
assert round(total_gib, 2) == 11.44
assert max_items == 10_485_760


### Try it 5. Choose GSI versus LSI from query patterns

**Task.** Decide the index type for four patterns: a customer's orders sorted by amount; all shipped orders across all customers; a customer's orders sorted by amount with strongly consistent reads; all orders by date added to a live table.

**Hint.** An LSI requires the table's partition key, cannot be added after creation, and is the only index that supports strongly consistent reads. Anything else is a GSI.

**Expected result.** `LSI`, `GSI`, `LSI`, `GSI` in that order.

In [ ]:
def index_type(same_partition_key, needs_strong_consistency, table_already_exists):
    if not same_partition_key:
        return "GSI"
    if needs_strong_consistency:
        return "LSI"
    return "GSI" if table_already_exists else "LSI"


patterns = [
    ("customer orders by amount", True, False, False),
    ("all shipped orders across customers", False, False, False),
    ("customer orders by amount, strongly consistent", True, True, False),
    ("global orders by date on a live table", False, False, True),
]

choices = [index_type(same, strong, exists) for _, same, strong, exists in patterns]
for (name, *_), choice in zip(patterns, choices):
    print(f"{name:<48} -> {choice}")
assert choices == ["LSI", "GSI", "LSI", "GSI"]


### Try it 6. Compute DAX savings from read patterns

**Task.** A service issues 8,000 reads/s of a 4 KB item, all eventually consistent and therefore cacheable. With a 65 percent item-cache hit rate and a 20 percent query-cache hit rate, compute the RCU/s with and without DAX and the percentage saved.

**Hint.** Only the misses reach the table; a 4 KB eventually consistent read costs half of one RCU.

**Expected result.** `4,000` RCU/s without DAX, `600` RCU/s with it, `85%` saved.

In [ ]:
def rcu_per_read(item_kb):
    return -(-item_kb // 4) / 2  # eventually consistent only, as DAX serves


reads_per_second = 8_000
item_cache_hit = 0.65
query_cache_hit = 0.20
missing = 1 - item_cache_hit - query_cache_hit

without_dax = reads_per_second * rcu_per_read(4)
with_dax = reads_per_second * missing * rcu_per_read(4)
saved_pct = (without_dax - with_dax) / without_dax

print(f"without DAX = {without_dax:,.0f} RCU/s")
print(f"with DAX    = {with_dax:,.0f} RCU/s")
print(f"saved       = {saved_pct:.0%}")
assert without_dax == 4000 and round(with_dax, 6) == 600
assert round(saved_pct, 2) == 0.85


### Try it 7. Plan a TTL and Streams pipeline

**Task.** 500 sessions are created per second, each 2 KiB, and expire after 7 days. Compute the steady-state item count and storage, the stream records per day if each session produces an INSERT and a later REMOVE, and decide whether the 24-hour stream retention is enough for a consumer that may be down for up to 7 days.

**Hint.** Steady state is `creates_per_second * ttl_days * 86_400`. A retention shorter than the maximum downtime means you need a durable sink.

**Expected result.** `302,400,000` items, about `576.8` GiB, `86,400,000` stream records per day, and the 1-day retention is not enough, so sink to S3.

In [ ]:
def steady_state_items(creates_per_second, ttl_days):
    return creates_per_second * ttl_days * 86_400


creates_per_second = 500
ttl_days = 7
items = steady_state_items(creates_per_second, ttl_days)
gib = items * 2048 / 1024 ** 3
records_per_day = creates_per_second * 86_400 * 2  # INSERT plus REMOVE
retention_days = 24 / 24
needs_sink = retention_days < ttl_days

print(f"steady-state sessions = {items:,}")
print(f"storage = {gib:.1f} GiB")
print(f"stream records/day = {records_per_day:,}")
print(f"retention {retention_days:.0f} day(s), sink to S3: {needs_sink}")
assert items == 302_400_000
assert round(gib, 1) == 576.8
assert records_per_day == 86_400_000
assert needs_sink is True
